In [13]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [3]:
import numpy as np
from scipy.linalg import lu, solve_triangular
import time
from scipy import sparse 

In [4]:
# -------------------------------
# Generate matrix and vector
# -------------------------------
#A = np.array([[2., 1., 1.],
#              [4., -6., 0.],
#              [-2., 7., 2.]])

#b = np.array([5., -2., 9.])
#n=A.shape[0]
#n = 3  # Matrix size

#A = np.random.rand(n, n)
#b = np.random.rand(n)


# Read system file and build sparse stiffness matrix
with np.load('./Dataset.npz') as system:
    u = system["x"]
    f = system["b"]
    K = sparse.coo_matrix(
        (system["A_values"], list(system["A_indices"])),
        shape=(u.size, u.size)
    )

# Check K * u = f
if(np.allclose(K.dot(u), f)):
    print("Everything is correct!")
else:
    print("There is something wrong!")
A_new = K.toarray()

b=f
n=A_new.shape[0]


Everything is correct!


In [5]:
# -------------------------------
# solve using LU Decomposition 
# -------------------------------
start = time.perf_counter()

# LU decomposition
P, L, U = lu(A)

# Forward substitution: Ly = Pb
y = solve_triangular(L, P @ b, lower=True)

# Back substitution: Ux = y
x = solve_triangular(U, y)

end = time.perf_counter()

elapsed = end - start


In [6]:
# -------------------------------
# FLOP estimation
# -------------------------------
# LU factorization      : (2/3)n^3
# Forward substitution  : n^2
# Back substitution     : n^2

flops = (2/3) * n**3 + 2 * n**2

performance = flops / elapsed
gflops = performance / 1e9

# -------------------------------
# Output
# -------------------------------
print(f"Matrix size              : {n} x {n}")
if(n<5):
	print("A =\n",A)
	print("P =\n", P)
	print("\nL =\n", L)
	print("\nU =\n", U)
	print("\nSolution x =")
	print(x)
print(f"Execution time           : {elapsed:.6f} seconds")
print(f"Estimated FLOPs          : {flops:.0f}")
print(f"Performance              : {performance:.3e} FLOPS")
print(f"Performance              : {gflops:.3e} GFLOPS")



Matrix size              : 1524 x 1524
Execution time           : 0.063414 seconds
Estimated FLOPs          : 2364382368
Performance              : 3.728e+10 FLOPS
Performance              : 3.728e+01 GFLOPS


In [7]:
#-----Solve using QR Decomposition----------------
start = time.perf_counter()

# QR decomposition
Q, R = np.linalg.qr(A)

# Compute Qᵀb
y = Q.T @ b

# Solve Rx = y
x = np.linalg.solve(R, y)

end = time.perf_counter()

elapsed = end - start

In [8]:
# ------------------------------------
# Performance calculations
# ------------------------------------
n = len(A)

# Approximate FLOPs
flops_qr = (4/3) * n**3          # QR decomposition
flops_qtb = 2 * n**2             # Matrix-vector multiplication
flops_backsolve = n**2           # Back substitution

total_flops = flops_qr + flops_qtb + flops_backsolve



gflops = total_flops / elapsed / 1e9

# ------------------------------------
# Output
# ------------------------------------
print(f"Matrix size              : {n} x {n}")
if(n<5):
    print("A =\n",A)
    print("Q =\n",Q)
    print("R =\n",R)
    print("Solution x =\n",x)


print(f"Execution time           : {elapsed:.6f} seconds")
print(f"Estimated FLOPs          : {total_flops:.0f}")
print(f"Performance              : {performance:.3e} FLOPS")
print(f"Performance              : {gflops:.3e} GFLOPS")
print("\nExecution time = {:.8f} s".format(elapsed))
print("Approximate FLOPs = {:.0f}".format(total_flops))
print("Performance = {:.6e} GFLOPS".format(gflops))

Matrix size              : 1524 x 1524
Execution time           : 0.138469 seconds
Estimated FLOPs          : 4726442160
Performance              : 3.728e+10 FLOPS
Performance              : 3.413e+01 GFLOPS

Execution time = 0.13846863 s
Approximate FLOPs = 4726442160
Performance = 3.413367e+01 GFLOPS
